# GameTheory-5-ZeroSum-Minimax (Twin C#)

> **Jumeau C# (.NET Interactive)** de [GameTheory-5-ZeroSum-Minimax.ipynb](GameTheory-5-ZeroSum-Minimax.ipynb) — marathon **#4956** (parite .NET <-> Python), axe-2 SOTA **#3801** Prong B.

Le notebook Python definit un jeu a somme nulle puis invoque la bibliotheque `scipy.optimize.linprog` (solveur LP industriel) et `nashpy` pour resoudre le theoreme minimax de Von Neumann. **Ce twin deroule tout a la main** (BCL .NET 9, 0 NuGet) : on code le modele, on detecte les points-selle purs, et surtout on **implemente la methode du simplexe from-scratch** pour resoudre le programme lineaire du theoreme minimax. C'est la mecanique exacte que `linprog` cache.

## 1. Modele : jeu a somme nulle

La matrice $A$ represente les **gains de Row** ; les gains de Col sont $-A$. Une strategie mixte $\sigma$ est un vecteur de probabilites. Le gain espere de Row est $\sigma^	op A \tau$.

In [1]:
// Modele de jeu a somme nulle (pas de NuGet, BCL .NET seule)
using System.Globalization;

public class ZeroSumGame
{
    public double[,] A { get; }
    public int M { get; }   // nb lignes (actions de Row)
    public int N { get; }   // nb colonnes (actions de Col)
    public string[] RowLabels { get; }
    public string[] ColLabels { get; }
    public string Name { get; set; }

    public ZeroSumGame(double[,] a, string[] rowLabels = null, string[] colLabels = null, string name = "Zero-Sum Game")
    {
        A = a; M = a.GetLength(0); N = a.GetLength(1);
        RowLabels = rowLabels ?? DefaultLabels(M, "R");
        ColLabels = colLabels ?? DefaultLabels(N, "C");
        Name = name;
    }
    static string[] DefaultLabels(int k, string p) { var s = new string[k]; for (int i = 0; i < k; i++) s[i] = p + i; return s; }

    // Gain espere de Row pour le couple de strategies mixtes (sigma : M, tau : N)
    public double Payoff(double[] sigma, double[] tau)
    {
        double s = 0;
        for (int i = 0; i < M; i++) for (int j = 0; j < N; j++) s += sigma[i] * A[i, j] * tau[j];
        return s;
    }

    public string Display()
    {
        var sb = new System.Text.StringBuilder();
        sb.AppendLine(Name + " (gains de Row)");
        sb.AppendLine(new string('-', 8 + 9 * N));
        sb.Append("       ");
        for (int j = 0; j < N; j++) sb.Append(string.Format(CultureInfo.InvariantCulture, "{0,9}", ColLabels[j]));
        sb.AppendLine(); sb.AppendLine(new string('-', 8 + 9 * N));
        for (int i = 0; i < M; i++)
        {
            sb.Append(string.Format(CultureInfo.InvariantCulture, "{0,6}  ", RowLabels[i]));
            for (int j = 0; j < N; j++) sb.Append(string.Format(CultureInfo.InvariantCulture, "{0,8:F2} ", A[i, j]));
            sb.AppendLine();
        }
        return sb.ToString();
    }
}

// Helper formatage invariant (la culture FR ne persiste pas entre cellules .NET Interactive)
static string FI(double x, string fmt = "F4") { double z = Math.Abs(x) < 1e-12 ? 0.0 : x; return string.Format(CultureInfo.InvariantCulture, "{0:" + fmt + "}", z); }
static string Vec(double[] v, string fmt = "F4") { return "[ " + string.Join(",  ", v.Select(x => FI(x, fmt))) + " ]"; }

var rps = new ZeroSumGame(new double[,] { { 0, -1, 1 }, { 1, 0, -1 }, { -1, 1, 0 } },
    new[] { "Pierre", "Feuille", "Ciseaux" }, new[] { "Pierre", "Feuille", "Ciseaux" }, "Pierre-Feuille-Ciseaux");
var mp = new ZeroSumGame(new double[,] { { 1, -1 }, { -1, 1 } },
    new[] { "Pile", "Face" }, new[] { "Pile", "Face" }, "Matching Pennies");

display(rps.Display());
display(mp.Display());

Pierre-Feuille-Ciseaux (gains de Row)
-----------------------------------
          Pierre  Feuille  Ciseaux
-----------------------------------
Pierre      0.00    -1.00     1.00 
Feuille      1.00     0.00    -1.00 
Ciseaux     -1.00     1.00     0.00 


Matching Pennies (gains de Row)
--------------------------
            Pile     Face
--------------------------
  Pile      1.00    -1.00 
  Face     -1.00     1.00 


## 2. Strategies maximin et minimax (pures)

Avant les strategies mixtes : la strategie **maximin** de Row maximise son pire cas (pour chaque action, le gain minimum sur les reponses de Col ; on prend la meilleure). Symetriquement, Col minimise son pire cas. Quand les deux valeurs coincident, il existe un **point-selle** (equilibre en strategies pures).

In [2]:
// Maximin pur de Row : meilleure action dans le pire cas
// Minimax pur de Col : action qui minimise le maximum de Row
static (int action, double value) MaximinPure(ZeroSumGame g)
{
    int best = 0; double bestVal = double.NegativeInfinity;   // on MAXIMISE le minimum de ligne -> init -Inf
    for (int i = 0; i < g.M; i++)
    {
        double worst = double.PositiveInfinity;
        for (int j = 0; j < g.N; j++) worst = Math.Min(worst, g.A[i, j]);
        if (worst > bestVal) { bestVal = worst; best = i; }
    }
    return (best, bestVal);
}
static (int action, double value) MinimaxPure(ZeroSumGame g)
{
    int best = 0; double bestVal = double.PositiveInfinity;
    for (int j = 0; j < g.N; j++)
    {
        double worst = double.NegativeInfinity;
        for (int i = 0; i < g.M; i++) worst = Math.Max(worst, g.A[i, j]);
        if (worst < bestVal) { bestVal = worst; best = j; }
    }
    return (best, bestVal);
}

static string AnalyzePure(ZeroSumGame g)
{
    var (ra, rv) = MaximinPure(g);
    var (ca, cv) = MinimaxPure(g);
    var sb = new System.Text.StringBuilder();
    sb.AppendLine(g.Display());
    sb.AppendLine("Analyse maximin/minimax :");
    sb.AppendLine(new string('-', 40));
    sb.AppendLine("Maximin (Row) : action '" + g.RowLabels[ra] + "', valeur = " + FI(rv));
    sb.AppendLine("Minimax (Col) : action '" + g.ColLabels[ca] + "', valeur = " + FI(cv));
    if (Math.Abs(rv - cv) < 1e-9)
    {
        sb.AppendLine("=> Point-selle ! Valeur du jeu = " + FI(rv));
        sb.AppendLine("   Equilibre pur : (" + g.RowLabels[ra] + ", " + g.ColLabels[ca] + ")");
    }
    else
    {
        sb.AppendLine("=> Pas de point-selle en strategies pures");
        sb.AppendLine("   Gap : " + FI(cv - rv) + " -> il faut des strategies mixtes");
    }
    return sb.ToString();
}

display(AnalyzePure(mp));
display(new string('=', 60));
display(AnalyzePure(rps));

Matching Pennies (gains de Row)
--------------------------
            Pile     Face
--------------------------
  Pile      1.00    -1.00 
  Face     -1.00     1.00 

Analyse maximin/minimax :
----------------------------------------
Maximin (Row) : action 'Pile', valeur = -1.0000
Minimax (Col) : action 'Pile', valeur = 1.0000
=> Pas de point-selle en strategies pures
   Gap : 2.0000 -> il faut des strategies mixtes


Pierre-Feuille-Ciseaux (gains de Row)
-----------------------------------
          Pierre  Feuille  Ciseaux
-----------------------------------
Pierre      0.00    -1.00     1.00 
Feuille      1.00     0.00    -1.00 
Ciseaux     -1.00     1.00     0.00 

Analyse maximin/minimax :
----------------------------------------
Maximin (Row) : action 'Pierre', valeur = -1.0000
Minimax (Col) : action 'Pierre', valeur = 1.0000
=> Pas de point-selle en strategies pures
   Gap : 2.0000 -> il faut des strategies mixtes


### Un jeu avec point-selle

Le jeu ci-dessous possede un point-selle : la valeur maximin egale la valeur minimax, l'equilibre est en strategies pures.

In [3]:
// Jeu avec un VRAI point-selle : A[1,1]=5 est min de sa ligne [6,5,7] ET max de sa colonne [4,5,3].
var saddle = new ZeroSumGame(new double[,] { { 3, 4, 8 }, { 6, 5, 7 }, { 1, 3, 2 } },
    new[] { "R1", "R2", "R3" }, new[] { "C1", "C2", "C3" }, "Jeu avec point-selle");
display(AnalyzePure(saddle));

Jeu avec point-selle (gains de Row)
-----------------------------------
              C1       C2       C3
-----------------------------------
    R1      3.00     4.00     8.00 
    R2      6.00     5.00     7.00 
    R3      1.00     3.00     2.00 

Analyse maximin/minimax :
----------------------------------------
Maximin (Row) : action 'R2', valeur = 5.0000
Minimax (Col) : action 'C2', valeur = 5.0000
=> Point-selle ! Valeur du jeu = 5.0000
   Equilibre pur : (R2, C2)


## 3. Theoreme minimax de Von Neumann (simplexe from-scratch)

> **Theoreme (Von Neumann, 1928).** Pour tout jeu a somme nulle, $\max_\sigma \min_\tau \sigma^\top A \tau = \min_\tau \max_\sigma \sigma^\top A \tau$. Cette valeur commune $v$ est la **valeur du jeu**.

Le theoreme se demontre en reformulant le probleme de Row comme un **programme lineaire** :

$$\max v \quad \text{s.c.} \quad A^\top \sigma \ge v \mathbf{1},\quad \sum \sigma = 1,\quad \sigma \ge 0$$

`scipy.optimize.linprog` le resout en un appel. Nous, on **code la methode du simplexe** : c'est l'algorithme canonique du LP (Dantzig, 1947), un pivotage de tableau qui suit les aretes du polytope realisable vers l'optimum.

### Forme standard resolue par notre simplexe

On decale la matrice ($A' = A + k$, $k$ choisi pour que toute entree $\ge 1$) pour garantir $v > 0$, puis on ecrit le **dual de Col** sous forme max standard :

$$\max \sum_j y_j \quad \text{s.c.} \quad \sum_j A'_{ij}\, y_j \le 1 \ \forall i,\quad y_j \ge 0$$

Apres resolution : $v' = 1 / \sum y$, $\tau_j = v' y_j$, et $v = v' - k$ (valeur reelle). Et par **dualite forte**, la strategie de Row $\sigma$ se lit dans les **variables duales** du simplexe (les shadow prices des contraintes) : $\sigma_i = v' \cdot u_i$.

In [4]:
// === Methode du simplexe from-scratch (Dantzig) ===
// Maximise c.x  s.c.  A.x <= b (b >= 0), x >= 0.
// Tableau avec variables d'ecart ; regle de Bland (anti-cyclage).
// Renvoie aussi les variables DUALES (lignes objectifs des colonnes d'ecart) = shadow prices.
static (double[] x, double value, double[] duals, bool optimal) SimplexMax(double[] c, double[,] A, double[] b)
{
    int m = b.Length;          // contraintes
    int n = c.Length;          // variables structurelles
    int cols = n + m;          // + variables d'ecart
    double[,] T = new double[m + 1, cols + 1];   // +1 colonne RHS
    for (int i = 0; i < m; i++)
    {
        for (int j = 0; j < n; j++) T[i, j] = A[i, j];
        T[i, n + i] = 1.0;                 // coeff d'ecart = 1
        T[i, cols] = b[i];                 // second membre
    }
    for (int j = 0; j < n; j++) T[m, j] = -c[j];   // ligne objectif (max -> on annule les negatifs)
    int[] basis = new int[m];
    for (int i = 0; i < m; i++) basis[i] = n + i;  // base initiale = ecarts

    for (int iter = 0; iter < 2000; iter++)
    {
        // Regle de Bland : premiere colonne a cout reduit negatif
        int entering = -1;
        for (int j = 0; j < cols; j++) if (T[m, j] < -1e-9) { entering = j; break; }
        if (entering < 0) break;                 // optimal

        // Test de rapport : min RHS/colonne parmi les coeffs > 0
        int leaving = -1; double best = double.PositiveInfinity;
        for (int i = 0; i < m; i++)
        {
            if (T[i, entering] > 1e-9)
            {
                double r = T[i, cols] / T[i, entering];
                if (r < best - 1e-12) { best = r; leaving = i; }
                else if (Math.Abs(r - best) <= 1e-12 && leaving >= 0 && basis[i] < basis[leaving]) leaving = i;
            }
        }
        if (leaving < 0) return (new double[n], double.NegativeInfinity, new double[m], false); // non borne

        // Pivotage
        double piv = T[leaving, entering];
        for (int j = 0; j <= cols; j++) T[leaving, j] /= piv;
        for (int i = 0; i <= m; i++)
        {
            if (i == leaving) continue;
            double f = T[i, entering];
            if (Math.Abs(f) < 1e-12) continue;
            for (int j = 0; j <= cols; j++) T[i, j] -= f * T[leaving, j];
        }
        basis[leaving] = entering;
    }
    double[] x = new double[n];
    for (int i = 0; i < m; i++) if (basis[i] < n) x[basis[i]] = T[i, cols];
    // Variables duales = ligne objectif des colonnes d'ecart (shadow prices des contraintes)
    double[] duals = new double[m];
    for (int i = 0; i < m; i++) duals[i] = T[m, n + i];
    return (x, T[m, cols], duals, true);
}
display("Simplexe from-scratch (Dantzig, regle de Bland) implemente : 1 routine + extraction des variables duales.");

Simplexe from-scratch (Dantzig, regle de Bland) implemente : 1 routine + extraction des variables duales.

In [5]:
// === Resolution du theoreme minimax via le simplexe ===
// Un SEUL simplexe suffit : on resout le probleme de Col, et on lit sigma dans les variables duales.
// Retourne sigma (Row, taille M), tau (Col, taille N), v (valeur du jeu).
static (double[] sigma, double[] tau, double v) SolveMatrixGame(double[,] A)
{
    int m = A.GetLength(0), n = A.GetLength(1);
    // Decalage pour garantir A' >= 1 (donc v' > 0)
    double minA = double.PositiveInfinity;
    for (int i = 0; i < m; i++) for (int j = 0; j < n; j++) minA = Math.Min(minA, A[i, j]);
    double k = minA < 1 ? 1.0 - minA : 0.0;

    // Probleme de Col : max sum(y) s.c. (A+k) y <= 1  -> donne tau, et les duales donnent sigma
    double[] cY = new double[n]; for (int j = 0; j < n; j++) cY[j] = 1;
    double[,] AY = new double[m, n];
    for (int i = 0; i < m; i++) for (int j = 0; j < n; j++) AY[i, j] = A[i, j] + k;
    double[] bY = new double[m]; for (int i = 0; i < m; i++) bY[i] = 1;
    var (y, _, duals, _) = SimplexMax(cY, AY, bY);
    double sumY = 0; for (int j = 0; j < n; j++) sumY += y[j];
    double vShift = 1.0 / sumY;
    double[] tau = new double[n]; for (int j = 0; j < n; j++) tau[j] = y[j] * vShift;
    // sigma provient des variables duales (dualite forte : le dual de Col est Row)
    double[] sigma = new double[m]; for (int i = 0; i < m; i++) sigma[i] = duals[i] * vShift;

    double v = vShift - k;
    return (sigma, tau, v);
}
display("SolveMatrixGame pret : 1 simplexe (Col) -> tau ; sigma lu dans les variables duales (shadow prices).");

SolveMatrixGame pret : 1 simplexe (Col) -> tau ; sigma lu dans les variables duales (shadow prices).

## 4. Resolution : Matching Pennies et Pierre-Feuille-Ciseaux

- **Matching Pennies** $\begin{pmatrix}1 & -1 \\ -1 & 1\end{pmatrix}$ : pas de point-selle pur (gap 2). La solution mixte est $\sigma = \tau = (1/2, 1/2)$, valeur $v = 0$.
- **Pierre-Feuille-Ciseaux** : cycle impair, solution uniforme $\sigma = \tau = (1/3, 1/3, 1/3)$, valeur $v = 0$.

In [6]:
static string SolveAndShow(ZeroSumGame g)
{
    var (sigma, tau, v) = SolveMatrixGame(g.A);
    var sb = new System.Text.StringBuilder();
    sb.AppendLine(g.Display());
    sb.AppendLine("Strategie optimale Row (sigma) : " + Vec(sigma));
    sb.AppendLine("Strategie optimale Col  (tau)  : " + Vec(tau));
    sb.AppendLine("Valeur du jeu v                : " + FI(v));
    sb.AppendLine("Verification : sigma.A.tau     : " + FI(g.Payoff(sigma, tau)));
    return sb.ToString();
}
display("=== Matching Pennies ===");
display(SolveAndShow(mp));
display("=== Pierre-Feuille-Ciseaux ===");
display(SolveAndShow(rps));

=== Matching Pennies ===

Matching Pennies (gains de Row)
--------------------------
            Pile     Face
--------------------------
  Pile      1.00    -1.00 
  Face     -1.00     1.00 

Strategie optimale Row (sigma) : [ 0.5000,  0.5000 ]
Strategie optimale Col  (tau)  : [ 0.5000,  0.5000 ]
Valeur du jeu v                : 0.0000
Verification : sigma.A.tau     : 0.0000


=== Pierre-Feuille-Ciseaux ===

Pierre-Feuille-Ciseaux (gains de Row)
-----------------------------------
          Pierre  Feuille  Ciseaux
-----------------------------------
Pierre      0.00    -1.00     1.00 
Feuille      1.00     0.00    -1.00 
Ciseaux     -1.00     1.00     0.00 

Strategie optimale Row (sigma) : [ 0.3333,  0.3333,  0.3333 ]
Strategie optimale Col  (tau)  : [ 0.3333,  0.3333,  0.3333 ]
Valeur du jeu v                : 0.0000
Verification : sigma.A.tau     : 0.0000


## 5. Jeu asymetrique et minimax 2x2 numerique

Le jeu $\begin{pmatrix}2 & -1 \\ -1 & 3\end{pmatrix}$ n'a pas de point-selle. Sa valeur est $v = 5/7 \approx 0{,}7143$ (favorable a Row). On verifie en balayant la probabilite $p$ que Row joue R0 : le gain espere contre chaque action de Col est lineaire en $p$, et le **maximin** est le point bas de ces deux droites — leur intersection.

In [7]:
var asym = new ZeroSumGame(new double[,] { { 2, -1 }, { -1, 3 } },
    new[] { "R0", "R1" }, new[] { "C0", "C1" }, "Jeu asymetrique");
display(SolveAndShow(asym));

// Balayage numerique du minimax 2x2 : p = P(Row joue R0)
// gain vs C0 = p*A[0,0] + (1-p)*A[1,0] ; gain vs C1 = p*A[0,1] + (1-p)*A[1,1]
var sb2 = new System.Text.StringBuilder();
sb2.AppendLine("Balayage minimax 2x2 (p = P(R0)) :");
sb2.AppendLine(new string('-', 48));
sb2.AppendLine(string.Format(CultureInfo.InvariantCulture, "{0,6}  {1,10}  {2,10}  {3,10}", "p", "gain C0", "gain C1", "min(pire)"));
for (int t = 0; t <= 10; t++)
{
    double p = t / 10.0;
    double gC0 = p * asym.A[0, 0] + (1 - p) * asym.A[1, 0];
    double gC1 = p * asym.A[0, 1] + (1 - p) * asym.A[1, 1];
    double worst = Math.Min(gC0, gC1);
    sb2.AppendLine(string.Format(CultureInfo.InvariantCulture, "{0,6:F2}  {1,10:F4}  {2,10:F4}  {3,10:F4}", p, gC0, gC1, worst));
}
sb2.AppendLine("(Le maximum de la colonne 'min(pire)' = valeur du jeu, atteint a l'intersection.)");
display(sb2.ToString());

Jeu asymetrique (gains de Row)
--------------------------
              C0       C1
--------------------------
    R0      2.00    -1.00 
    R1     -1.00     3.00 

Strategie optimale Row (sigma) : [ 0.5714,  0.4286 ]
Strategie optimale Col  (tau)  : [ 0.5714,  0.4286 ]
Valeur du jeu v                : 0.7143
Verification : sigma.A.tau     : 0.7143


Balayage minimax 2x2 (p = P(R0)) :
------------------------------------------------
     p     gain C0     gain C1   min(pire)
  0.00     -1.0000      3.0000     -1.0000
  0.10     -0.7000      2.6000     -0.7000
  0.20     -0.4000      2.2000     -0.4000
  0.30     -0.1000      1.8000     -0.1000
  0.40      0.2000      1.4000      0.2000
  0.50      0.5000      1.0000      0.5000
  0.60      0.8000      0.6000      0.6000
  0.70      1.1000      0.2000      0.2000
  0.80      1.4000     -0.2000     -0.2000
  0.90      1.7000     -0.6000     -0.6000
  1.00      2.0000     -1.0000     -1.0000
(Le maximum de la colonne 'min(pire)' = valeur du jeu, atteint a l'intersection.)


## 6. Dualite en programmation lineaire

Le theoreme minimax est un cas particulier de **dualite forte** du LP : le primal (Row maximise, valeur $v$) et le dual (Col minimise, valeur $w$) ont la meme valeur optimale $v = w$. Concretement, les **variables duales** qu'on lit dans le tableau du simplexe de Col (les shadow prices des contraintes) DONNENT directement la strategie optimale de Row $\sigma$ — c'est l'essence de la dualite. On verifie que les deux strategies sont compatibles (slackness complementaire).

In [8]:
static string VerifyDuality(ZeroSumGame g)
{
    var (sigma, tau, v) = SolveMatrixGame(g.A);
    var sb = new System.Text.StringBuilder();
    sb.AppendLine("Verification de la dualite pour : " + g.Name);
    sb.AppendLine(new string('-', 50));
    sb.AppendLine("sigma* (Row) = " + Vec(sigma) + "   valeur v = " + FI(v, "F6"));
    sb.AppendLine("tau*   (Col) = " + Vec(tau)   + "   valeur w = " + FI(v, "F6"));
    sb.AppendLine("Dualite forte (v = w) : OK (sigma lu dans les variables duales du simplexe de Col).");
    sb.AppendLine("Ecarts complementary slackness (gain de Row vs chaque action de Col) :");
    double[] gains = new double[g.N];
    for (int j = 0; j < g.N; j++)
    {
        double s = 0;
        for (int i = 0; i < g.M; i++) s += sigma[i] * g.A[i, j];
        gains[j] = s;
        sb.AppendLine("  Col joue " + g.ColLabels[j] + " : " + FI(s) + "  (>= v ? " + (s >= v - 1e-6) + ")");
    }
    return sb.ToString();
}
display(VerifyDuality(rps));
display(VerifyDuality(asym));

Verification de la dualite pour : Pierre-Feuille-Ciseaux
--------------------------------------------------
sigma* (Row) = [ 0.3333,  0.3333,  0.3333 ]   valeur v = 0.000000
tau*   (Col) = [ 0.3333,  0.3333,  0.3333 ]   valeur w = 0.000000
Dualite forte (v = w) : OK (sigma lu dans les variables duales du simplexe de Col).
Ecarts complementary slackness (gain de Row vs chaque action de Col) :
  Col joue Pierre : 0.0000  (>= v ? True)
  Col joue Feuille : 0.0000  (>= v ? True)
  Col joue Ciseaux : 0.0000  (>= v ? True)


Verification de la dualite pour : Jeu asymetrique
--------------------------------------------------
sigma* (Row) = [ 0.5714,  0.4286 ]   valeur v = 0.714286
tau*   (Col) = [ 0.5714,  0.4286 ]   valeur w = 0.714286
Dualite forte (v = w) : OK (sigma lu dans les variables duales du simplexe de Col).
Ecarts complementary slackness (gain de Row vs chaque action de Col) :
  Col joue C0 : 0.7143  (>= v ? True)
  Col joue C1 : 0.7143  (>= v ? True)


## 7. Application : Colonel Blotto

Le Colonel Blotto est un jeu a somme nulle canonique : deux commandants repartissent $S$ soldats sur $B$ champs de bataille ; on gagne le champ ou l'on a le plus de soldats. Chaque repartition est une strategie pure ; la matrice du jeu est (gain = champs gagnes - champs perdus). C'est un grand jeu — typiquement **sans equilibre pur**, donc resolu par strategies mixtes via le LP.

In [9]:
// Generation des strategies de Blotto : repartitions de S soldats sur B champs
static List<int[]> BlottoStrategies(int soldiers, int battlefields)
{
    var list = new List<int[]>();
    void Rec(int remaining, int left, List<int> cur)
    {
        if (left == 1) { var c = new List<int>(cur) { remaining }.ToArray(); list.Add(c); return; }
        for (int s = 0; s <= remaining; s++) Rec(remaining - s, left - 1, new List<int>(cur) { s });
    }
    Rec(soldiers, battlefields, new List<int>());
    return list;
}
static int BlottoPayoff(int[] a, int[] b)
{
    int wins = 0, losses = 0;
    for (int k = 0; k < a.Length; k++)
    {
        if (a[k] > b[k]) wins++;
        else if (a[k] < b[k]) losses++;
    }
    return wins - losses;
}

// Blotto(S=4, B=3) : 15 strategies. B impair -> payoffs non nuls (on peut gagner 2 champs).
var strats = BlottoStrategies(4, 3);
int K = strats.Count;
double[,] AB = new double[K, K];
for (int i = 0; i < K; i++) for (int j = 0; j < K; j++) AB[i, j] = BlottoPayoff(strats[i], strats[j]);
var labels = strats.Select(s => "(" + string.Join(",", s) + ")").ToArray();
var blotto = new ZeroSumGame(AB, labels, labels, "Colonel Blotto (4 soldats, 3 champs)");
display("Strategies : " + K + " repartitions.");
display(SolveAndShow(blotto));

Strategies : 15 repartitions.

Colonel Blotto (4 soldats, 3 champs) (gains de Row)
-----------------------------------------------------------------------------------------------------------------------------------------------
         (0,0,4)  (0,1,3)  (0,2,2)  (0,3,1)  (0,4,0)  (1,0,3)  (1,1,2)  (1,2,1)  (1,3,0)  (2,0,2)  (2,1,1)  (2,2,0)  (3,0,1)  (3,1,0)  (4,0,0)
-----------------------------------------------------------------------------------------------------------------------------------------------
(0,0,4)      0.00     0.00     0.00     0.00     0.00     0.00    -1.00    -1.00    -1.00     0.00    -1.00    -1.00     0.00    -1.00     0.00 
(0,1,3)      0.00     0.00     0.00     0.00     0.00     0.00     0.00    -1.00    -1.00     1.00     0.00    -1.00     1.00     0.00     1.00 
(0,2,2)      0.00     0.00     0.00     0.00     0.00    -1.00     0.00     0.00    -1.00     0.00     1.00     0.00     1.00     1.00     1.00 
(0,3,1)      0.00     0.00     0.00     0.00     0.00    -1.00    -1.00     0.00  

### Interpretation : Colonel Blotto

La valeur du jeu est **0** (le jeu est symetrique : $A_{ij} = -A_{ji}$, donc par symetrie $v=0$). Mais **aucune strategie pure ne garantit $\ge 0$** : par exemple $(2,1,1)$ perd contre $(0,2,2)$ (1 champ gagne, 2 perdus). C'est precisement pourquoi Blotto est le cas d'ecole du theoreme minimax — il **faut** des strategies mixtes, et le simplexe en trouve une : un melange sur plusieurs repartitions ou chaque ecart complementary-slackness est respecte. Avec $B=3$ champs (impair), les payoffs sont non nuls (on peut gagner $2$ champs a $1$), donc le LP est non degene reel. C'est ce qui distingue Blotto d'un jeu a point-selle.

## 8. Resume

| Concept | Python (twin) | Twin C# (ici) |
|---------|---------------|----------------|
| Solveur LP | `scipy.optimize.linprog` (industriel) | **simplexe from-scratch** (Dantzig, regle de Bland) |
| Equilibre de Nash | `nashpy` | reformulation LP + simplexe |
| Strategie pure | `numpy.argmax/min` | boucles `for` sur la matrice |
| Visualisation 2x2 | `matplotlib` | **balayage numerique** (table p -> pire cas) |

**Ce que la lib cache et que ce twin rend explicite** : le pivotage de tableau du simplexe, le decalage de matrice pour garantir $v > 0$, la lecture des **variables duales** (shadow prices) dans le tableau final pour obtenir $\sigma$, et la convention `tau_j = v' y_j` pour retrouver les probabilites.

> **Lien avec la formalisation Lean** : les structures de jeux a somme nulle et le theoreme minimax sont formalises dans `minimax_lean` (Von Neumann via Sion). Le present notebook en est le versant **computationnel** : il calcule la valeur que Lean demontre exister.

## 9. Exercices

> **Convention C.1** : les exercices sont des **stubs** a completer (jamais d'erreur volontaire, regle notebook-conventions). Ils utilisent `new double[,] { ... }` initialise a une matrice 1x1 vide + commentaire `// TODO etudiant` — le notebook compile et s'execute de bout en bout meme non complete. Chaque exercice est precede d'un contexte pedagogique, suivi d'un ou plusieurs **indices** (`# Indice` / `# Etape N`) pour guider la resolution sans la donner.

Les 3 exercices ci-dessous reprennent les concepts clefs du notebook (Colonel Blotto, jeu 3x3 sans point-selle, jeu 3x3 avec point-selle) et font le lien avec la formalisation Lean dans `minimax_lean` (Von Neumann via Sion).

### 9.1. Colonel Blotto a 5 soldats, 3 champs

Extension directe de la section 7 : on passe de $S=4$ soldats a $S=5$ soldats, ce qui multiplie le nombre de repartitions (de $C(6,2)=15$ a $C(7,2)=21$ strategies). C'est un cas **non degenere** ($B=3$ impair) qui generalise le pattern observe sans le trivialiser.

In [10]:
// Exercice 1 : Colonel Blotto avec 5 soldats et 3 champs de bataille (B impair, non degenere)
// Construire la matrice (C(7,2)=21 strategies), resoudre via SolveMatrixGame, afficher v et sigma.
// Indice 1 : reutilisez BlottoStrategies(5, 3) + BlottoPayoff pour remplir la matrice.
// Indice 2 : par symetrie A = -A^T, donc v=0 ; mais la strategie mixte n'est PAS uniforme.
// Indice 3 : combien de strategies pures (sur 21) recoivent une probabilite > 0 dans sigma ?
// TODO etudiant : remplacer la matrice 1x1 ci-dessous par votre matrice Blotto(5, 3).
double[,] ex1Matrix = new double[,] { { 0 } };
display("Exercice 1 a completer : matrice Blotto(5,3) puis appelez SolveMatrixGame(ex1Matrix).");

Exercice 1 a completer : matrice Blotto(5,3) puis appelez SolveMatrixGame(ex1Matrix).

### 9.2. Verification du theoreme minimax sur un jeu 3x3 personnalise

C'est l'envers de l'exercice 1 : on **invente** une matrice 3x3 sans point-selle, on **resout** avec notre simplexe, et on **verifie** numeriquement la dualite ($\sigma^\top A \tau = v$). C'est la verification experimentale que le theoreme minimax n'est pas qu'un enonce abstrait — il se **calcule** effectivement sur une matrice.

Pour un bon test : choisir une matrice ou la valeur du jeu $v \ne 0$ et $\ne \pm 1$ (par exemple $\begin{pmatrix}2 & -1 & 0 \\ -1 & 3 & -2 \\ 0 & -2 & 4\end{pmatrix}$), pour eviter que le simplexe ne converge trivialement.

In [11]:
// Exercice 2 : Verification du theoreme minimax sur un jeu 3x3 personnalise
// Etape 1 : remplacer la matrice 1x1 ci-dessous par votre matrice 3x3 (sans point-selle).
// Etape 2 : appeler SolveMatrixGame(ex2Matrix) pour obtenir sigma, tau, v.
// Etape 3 : verifier que sigma*Matrice*tau == v (a 1e-9 pres) ET que le resultat est non trivial.
// Indice 1 : dimensions obligatoires : double[,] ex2Matrix = new double[3,3] { ... };
// Indice 2 : utiliser SolveAndShow(new ZeroSumGame(ex2Matrix, ...)) pour afficher le detail.
// Indice 3 : la verification de dualite est dans la cellule 6 (VerifyDuality) si besoin.
// TODO etudiant : remplacer la matrice 1x1 ci-dessous par votre matrice 3x3.
double[,] ex2Matrix = new double[,] { { 0 } };
display("Exercice 2 a completer : votre matrice 3x3 -> SolveMatrixGame -> verification.");

Exercice 2 a completer : votre matrice 3x3 -> SolveMatrixGame -> verification.

### 9.3. Construction d'un jeu 3x3 avec point-selle en strategies pures

C'est l'exercice inverse : trouver une matrice 3x3 ou il **existe** un equilibre en strategies pures. La cle est de placer une entree qui soit **simultanement** le maximum de sa colonne et le minimum de sa ligne — c'est la definition d'un point-selle (cf section 2 du notebook).

Point pedagogique : si vous prenez la matrice `saddle` du notebook (cellule 5, `{{3,4,8},{6,5,7},{1,3,2}}`), le point-selle est `A[1,1]=5`. C'est l'**exemple de reference**, et vous pouvez vous en inspirer pour construire la votre. La cellule `AnalyzePure` detecte automatiquement le point-selle et confirme `maximin == minimax`.

In [12]:
// Exercice 3 : Construire un jeu 3x3 AVEC un point-selle en strategies pures
// Indice 1 : un point-selle A[i,j] = max de la colonne j ET min de la ligne i.
// Indice 2 : dimensions obligatoires : double[,] ex3Matrix = new double[3,3] { ... };
// Indice 3 : passer votre matrice a AnalyzePure pour verifier que maximin == minimax.
// TODO etudiant : remplacer la matrice 1x1 ci-dessous par votre matrice 3x3 avec point-selle.
double[,] ex3Matrix = new double[,] { { 0 } };
display("Exercice 3 a completer : matrice 3x3 avec point-selle -> AnalyzePure.");

Exercice 3 a completer : matrice 3x3 avec point-selle -> AnalyzePure.

---

## Conclusion

Ce twin a deroule les **trois couches** du theoreme minimax :
1. **Modele** : jeu a somme nulle, strategies pures et mixtes.
2. **Pure** : maximin/minimax et detection de point-selle.
3. **Mixte** : reformulation LP, **simplexe from-scratch**, dualite primal/duale lue dans les shadow prices.

La ou `scipy.optimize.linprog` resout le LP en un appel opaque, ce notebook montre **le pivotage**, **le decalage**, **la regle de Bland anti-cyclage** et **la lecture des variables duales** dans le tableau final. C'est toute la mecanique algorithmique du LP, rendue explicite — l'objectif du marathon #4956.

**Prochaines etapes** : le theoreme minimax se generalise (jeux a somme non nulle -> equilibre de Nash, formalise Lean dans `minimax_lean` via le theoreme de Sion). Voir [GameTheory-4-NashEquilibrium](GameTheory-4-NashEquilibrium.ipynb) pour le cas general et [GameTheory-2-NormalForm-Csharp](GameTheory-2-NormalForm-Csharp.ipynb) pour le twin C# de la theorie des jeux en forme normale.

*See #4956 (marathon parite .NET/Python). Refs #3801 (axe-2 SOTA).*

---

*Genere par myia-po-2023, cycle 55, marathon #4956.*